In [ ]:
!pip -q install pytorchvideo av fvcore iopath


In [ ]:
from __future__ import annotations

import json
import shutil
import tarfile
import zipfile
from contextlib import nullcontext
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.amp import autocast
from tqdm.auto import tqdm

from google.colab import drive
from pytorchvideo.models.hub import slowfast_r50

drive.mount('/content/drive')

STA_ROOT = Path('/content/drive/MyDrive/momentlens_datasets/charades_sta_3fps_v1')
STA_ANNOT_DIR = STA_ROOT / 'data' / 'annotations'
STA_SPLIT_DIR = STA_ROOT / 'splits'

RAW_VIDEO_ROOT = Path('/content/charades_raw_from_drive')
RAW_VIDEO_ARCHIVE_ROOT = Path('/content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch')
RAW_VIDEO_ZIP = RAW_VIDEO_ARCHIVE_ROOT / 'downloads' / 'Charades_v1_480.zip'
VIDEO_ROOT = RAW_VIDEO_ROOT / 'raw' / 'videos_480p' / 'Charades_v1_480'

PHASE1_RUN_ROOT = Path('/content/drive/MyDrive/momentlens_runs/charades_slowfast_phase1/run_001')
BEST_CKPT = PHASE1_RUN_ROOT / 'best.pt'

LOCAL_PHASE2_RUN_ROOT = Path('/content/momentlens_runs/charades_slowfast_phase2/run_003')
DRIVE_PHASE2_RUN_ROOT = Path('/content/drive/MyDrive/momentlens_runs/charades_slowfast_phase2/run_003')
FEATURE_ROOT = LOCAL_PHASE2_RUN_ROOT / 'features'
LOCAL_PHASE2_RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_PHASE2_RUN_ROOT.mkdir(parents=True, exist_ok=True)
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_RUNTIME_ROOT = Path('/content/charades_sta_3fps_v1')
ANNOT_DIR = LOCAL_RUNTIME_ROOT / 'data' / 'annotations'
SPLIT_DIR = LOCAL_RUNTIME_ROOT / 'splits'


TRAIN_ANNOT_PATH = ANNOT_DIR / 'train.json'
VAL_ANNOT_PATH = ANNOT_DIR / 'val.json'
TEST_ANNOT_PATH = ANNOT_DIR / 'test.json'

FEATURE_CLIP_LEN_SEC = 1.0
VIDEO_TIMELINE_FPS = 24.0
NUM_FRAMES = 32
ALPHA = 4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

KINETICS_MEAN = torch.tensor([0.45, 0.45, 0.45]).view(1, 3, 1, 1)
KINETICS_STD = torch.tensor([0.225, 0.225, 0.225]).view(1, 3, 1, 1)

LOCAL_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
ANNOT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
RAW_VIDEO_ROOT.mkdir(parents=True, exist_ok=True)
VIDEO_ROOT.parent.mkdir(parents=True, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# def copy_if_missing(src: Path, dst: Path):
#     if not src.exists():
#         raise FileNotFoundError(src)
#     dst.parent.mkdir(parents=True, exist_ok=True)
#     if dst.exists() and dst.stat().st_size == src.stat().st_size:
#         return
#     shutil.copy2(src, dst)


# copy_if_missing(STA_ANNOT_DIR / 'train.json', TRAIN_ANNOT_PATH)
# copy_if_missing(STA_ANNOT_DIR / 'val.json', VAL_ANNOT_PATH)
# copy_if_missing(STA_ANNOT_DIR / 'test.json', TEST_ANNOT_PATH)
# copy_if_missing(STA_SPLIT_DIR / 'train_video_ids.txt', SPLIT_DIR / 'train_video_ids.txt')
# copy_if_missing(STA_SPLIT_DIR / 'val_video_ids.txt', SPLIT_DIR / 'val_video_ids.txt')

# if not RAW_VIDEO_ZIP.exists():
#     raise FileNotFoundError(RAW_VIDEO_ZIP)
# if not VIDEO_ROOT.exists():
#     with zipfile.ZipFile(RAW_VIDEO_ZIP, 'r') as zf:
#         zf.extractall(VIDEO_ROOT.parent)


In [ ]:
with TRAIN_ANNOT_PATH.open('r', encoding='utf-8') as f:
    train_samples = json.load(f)
with VAL_ANNOT_PATH.open('r', encoding='utf-8') as f:
    val_samples = json.load(f)
with TEST_ANNOT_PATH.open('r', encoding='utf-8') as f:
    test_samples = json.load(f)

def unique_video_rows(samples):
    rows = []
    seen = set()
    for row in samples:
        video_id = row['video_id']
        if video_id in seen:
            continue
        seen.add(video_id)
        rows.append({**row, 'video_path': str((VIDEO_ROOT / f'{video_id}.mp4').resolve())})
    return rows

train_rows = unique_video_rows(train_samples)
val_rows = unique_video_rows(val_samples)
test_rows = unique_video_rows(test_samples)

if not BEST_CKPT.exists():
    raise FileNotFoundError(BEST_CKPT)

print(f'loaded STA annotations: train_videos={len(train_rows)} val_videos={len(val_rows)} test_videos={len(test_rows)}')
print(f'feature source checkpoint: {BEST_CKPT}')


loaded STA annotations: train_videos=4270 val_videos=1068 test_videos=1334
feature source checkpoint: /content/drive/MyDrive/momentlens_runs/charades_slowfast_phase1/run_001/best.pt


In [ ]:
with (SPLIT_DIR / 'train_video_ids.txt').open('r', encoding='utf-8') as f:
    train_video_ids_ref = [line.strip() for line in f if line.strip()]
with (SPLIT_DIR / 'val_video_ids.txt').open('r', encoding='utf-8') as f:
    val_video_ids_ref = [line.strip() for line in f if line.strip()]

train_video_ids = sorted({row['video_id'] for row in train_samples})
val_video_ids = sorted({row['video_id'] for row in val_samples})
test_video_ids = sorted({row['video_id'] for row in test_samples})

if set(train_video_ids) != set(train_video_ids_ref):
    raise RuntimeError(f'train video_id mismatch: samples={len(train_video_ids)} split_file={len(train_video_ids_ref)}')
if set(val_video_ids) != set(val_video_ids_ref):
    raise RuntimeError(f'val video_id mismatch: samples={len(val_video_ids)} split_file={len(val_video_ids_ref)}')

train_val_overlap = set(train_video_ids) & set(val_video_ids)
train_test_overlap = set(train_video_ids) & set(test_video_ids)
val_test_overlap = set(val_video_ids) & set(test_video_ids)
if train_val_overlap or train_test_overlap or val_test_overlap:
    raise RuntimeError('STA split overlap detected')

expected_video_ids = sorted(set(train_video_ids) | set(val_video_ids) | set(test_video_ids))
missing_video_ids = [video_id for video_id in expected_video_ids if not (VIDEO_ROOT / f'{video_id}.mp4').exists()]
if missing_video_ids:
    raise RuntimeError(f'missing raw videos: {len(missing_video_ids)} first={missing_video_ids[:10]}')

print('STA video check passed')
print(f'train videos: {len(train_video_ids)}')
print(f'val videos: {len(val_video_ids)}')
print(f'test videos: {len(test_video_ids)}')
print(f'used videos total: {len(expected_video_ids)}')
print('missing raw videos: 0')


STA video check passed
train videos: 4270
val videos: 1068
test videos: 1334
used videos total: 6672
missing raw videos: 0


In [ ]:
def sample_windows(video_len_sec):
    full_clips = int(float(video_len_sec) // FEATURE_CLIP_LEN_SEC)
    if full_clips <= 0:
        return [(0.0, float(video_len_sec))]

    return [(float(i * FEATURE_CLIP_LEN_SEC), float((i + 1) * FEATURE_CLIP_LEN_SEC)) for i in range(full_clips)]


def decode_video_timeline(video_path, timeline_fps=VIDEO_TIMELINE_FPS):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'cannot open video: {video_path}')

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 24.0)
    step_sec = 1.0 / float(timeline_fps)
    next_sample_sec = 0.0
    frame_idx = 0
    frames = []

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        time_sec = frame_idx / fps
        if time_sec + 1e-8 >= next_sample_sec:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224), interpolation=cv2.INTER_AREA)
            frames.append(torch.from_numpy(frame).permute(2, 0, 1).contiguous().to(torch.uint8))
            next_sample_sec += step_sec
        frame_idx += 1

    cap.release()

    if not frames:
        raise RuntimeError(f'No frames decoded from: {video_path}')

    return torch.stack(frames)


def decode_window_from_timeline(frames, clip_start, clip_end, num_frames=NUM_FRAMES):
    window_len = 24
    window_len = max(1, window_len)

    start_idx = int(round(clip_start * VIDEO_TIMELINE_FPS))
    max_start_idx = max(0, frames.shape[0] - window_len)
    start_idx = max(0, min(start_idx, max_start_idx))
    clip_frames = frames[start_idx:start_idx + window_len]

    if clip_frames.shape[0] == 0:
        nearest = min(max(start_idx, 0), frames.shape[0] - 1)
        clip_frames = frames[nearest:nearest + 1]

    if clip_frames.shape[0] < num_frames:
        pad = clip_frames[-1:].repeat(num_frames - clip_frames.shape[0], 1, 1, 1)
        clip_frames = torch.cat([clip_frames, pad], dim=0)
    elif clip_frames.shape[0] > num_frames:
        idx = np.linspace(0, clip_frames.shape[0] - 1, num_frames).astype(np.int64)
        clip_frames = clip_frames[idx]

    clip_frames = clip_frames.to(torch.float32).div(255.0)
    clip_frames = (clip_frames - KINETICS_MEAN) / KINETICS_STD
    return clip_frames


In [ ]:
feature_model = slowfast_r50(pretrained=False)
head = feature_model.blocks[-1]
if not hasattr(head, 'proj'):
    raise RuntimeError('Unexpected SlowFast head layout')
head.proj = nn.Linear(2304, 157)
feature_model.load_state_dict(torch.load(BEST_CKPT, map_location='cpu'))
head.proj = nn.Identity()
feature_model = feature_model.to(DEVICE).eval()
for p in feature_model.parameters():
    p.requires_grad = False

amp_ctx = autocast('cuda') if DEVICE == 'cuda' else nullcontext()


In [ ]:
def extract_video_features(row):
    frames = decode_video_timeline(row['video_path'])
    windows = sample_windows(float(row['duration']))
    clips = []
    starts = []
    ends = []
    for start_sec, end_sec in windows:
        clips.append(decode_window_from_timeline(frames, start_sec, end_sec))
        starts.append(start_sec)
        ends.append(end_sec)

    clip_batch = torch.stack(clips)  # [N, T, C, H, W]
    fast = clip_batch.permute(0, 2, 1, 3, 4).contiguous()
    slow = fast[:, :, ::ALPHA, :, :].contiguous()

    with torch.inference_mode():
        with amp_ctx:
            feats = feature_model([slow.to(DEVICE, non_blocking=True), fast.to(DEVICE, non_blocking=True)])

    return feats.detach().cpu().to(torch.float32), torch.tensor(starts, dtype=torch.float32), torch.tensor(ends, dtype=torch.float32)


feature_manifest = {
    'source_checkpoint': str(BEST_CKPT),
    'feature_clip_len_sec': FEATURE_CLIP_LEN_SEC,
    'num_frames': NUM_FRAMES,
    'video_timeline_fps': VIDEO_TIMELINE_FPS,
    'alpha': ALPHA,
    'device': DEVICE,
    'video_source': 'charades_raw_from_scratch_480p',
    'split_source': 'charades_sta_3fps_v1',
    'splits': {},
}

split_jobs = [('train', train_rows), ('val', val_rows), ('test', test_rows)]
for split_name, rows in split_jobs:
    split_dir = FEATURE_ROOT / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    split_entries = []
    for row in tqdm(rows, desc=f'extract {split_name}', leave=True):
        out_path = split_dir / f"{row['video_id']}.pt"
        if out_path.exists():
            split_entries.append({
                'video_id': row['video_id'],
                'split': split_name,
                'feature_path': str(out_path),
            })
            continue

        feats, clip_starts, clip_ends = extract_video_features(row)
        payload = {
            'video_id': row['video_id'],
            'split': split_name,
            'video_path': row['video_path'],
            'video_length_sec': float(row['duration']),
            'clip_starts': clip_starts,
            'clip_ends': clip_ends,
            'features': feats,
            'feature_dim': int(feats.shape[-1]),
            'num_windows': int(feats.shape[0]),
            'source_checkpoint': str(BEST_CKPT),
            'feature_clip_len_sec': FEATURE_CLIP_LEN_SEC,
                    'num_frames': NUM_FRAMES,
            'video_timeline_fps': VIDEO_TIMELINE_FPS,
        }
        torch.save(payload, out_path)
        split_entries.append({
            'video_id': row['video_id'],
            'split': split_name,
            'feature_path': str(out_path),
        })
    feature_manifest['splits'][split_name] = split_entries

LOCAL_FEATURE_MANIFEST = LOCAL_PHASE2_RUN_ROOT / 'feature_manifest.json'
DRIVE_FEATURE_MANIFEST = DRIVE_PHASE2_RUN_ROOT / 'feature_manifest.json'
with LOCAL_FEATURE_MANIFEST.open('w', encoding='utf-8') as f:
    json.dump(feature_manifest, f, indent=2)

FEATURE_TAR_LOCAL = LOCAL_PHASE2_RUN_ROOT / 'features.tar'
FEATURE_TAR_DRIVE = DRIVE_PHASE2_RUN_ROOT / 'features.tar'
with tqdm(total=1, desc='tar features', leave=True) as pbar:
    with tarfile.open(FEATURE_TAR_LOCAL, 'w') as tf:
        for file_path in sorted(FEATURE_ROOT.rglob('*')):
            if file_path.is_file():
                tf.add(file_path, arcname=file_path.relative_to(LOCAL_PHASE2_RUN_ROOT))
    pbar.update(1)

shutil.copy2(LOCAL_FEATURE_MANIFEST, DRIVE_FEATURE_MANIFEST)
shutil.copy2(FEATURE_TAR_LOCAL, FEATURE_TAR_DRIVE)

print('feature extraction done')
print(f'local feature tar: {FEATURE_TAR_LOCAL}')
print(f'drive feature tar: {FEATURE_TAR_DRIVE}')


extract train:   0%|          | 0/4270 [00:00<?, ?it/s]

extract val:   0%|          | 0/1068 [00:00<?, ?it/s]

extract test:   0%|          | 0/1334 [00:00<?, ?it/s]

tar features:   0%|          | 0/1 [00:00<?, ?it/s]

feature extraction done
local feature tar: /content/momentlens_runs/charades_slowfast_phase2/run_003/features.tar
drive feature tar: /content/drive/MyDrive/momentlens_runs/charades_slowfast_phase2/run_003/features.tar


In [ ]:
sample_feature_path = FEATURE_ROOT / 'train' / f"{train_rows[0]['video_id']}.pt"
sample = torch.load(sample_feature_path, map_location='cpu')
print(f"sample feature path: {sample_feature_path}")
print(f"features shape: {tuple(sample['features'].shape)} dtype={sample['features'].dtype}")
print(f"windows: {sample['num_windows']} feature_dim={sample['feature_dim']}")


sample feature path: /content/momentlens_runs/charades_slowfast_phase2/run_003/features/train/AO8RW.pt
features shape: (34, 2304) dtype=torch.float32
windows: 34 feature_dim=2304
